In [1]:
import numpy as np
import pandas as pd
import scipy
import re
import os
from pathlib import Path

Raw datasets downloaded from https://www.elsevier.com/products/scopus/content

on 28 October 2026

(according to Scopus, last updated Sep 2026)

In [3]:
# current sources

current_df = pd.read_excel('../data/261028_scopus/ext_list_Sep_2026.xlsx', sheet_name='Scopus Sources Sep. 2026')

In [4]:
current_df['Active or Inactive'].value_counts()

Active      32260
Inactive    16844
Name: Active or Inactive, dtype: int64

In [5]:
current_df = current_df[current_df['Active or Inactive'] == 'Active'].copy()

In [7]:
# past sources

past_df = pd.read_excel('../data/261028_scopus/ext_list_Sep_2026.xlsx', 
                        sheet_name='Discontinued Titles Sep. 2026', header=1)

In [8]:
current_df['indexing_status'] = 'indexed'
current_df['journal_title'] = current_df['Source Title']
past_df['indexing_status'] = 'deindexed'
past_df['journal_title'] = past_df['Source Title (newly added titles are highlighted in red)']
past_df['notes_on_indexing_status'] = past_df['Indexation Change']

In [10]:
# conference proceedings

conf_df = pd.read_excel('../data/261028_scopus/ext_list_Sep_2026.xlsx', 
                        sheet_name='Serial Conf. Proc. with Profile', header=0)

def mapper(f):
    if str(f) == 'nan':
        return 'indexed'
    else:
        return 'deindexed'
    
conf_df['indexing_status'] = conf_df['Titles Discontinued by Scopus'].apply(mapper)
conf_df['journal_title'] = conf_df['Source Title']
conf_df['notes_on_indexing_status'] = 'serial conference proceedings'

In [11]:
df_all = pd.concat([current_df, past_df, conf_df])

In [12]:
def process_issn(issn):
    issn = str(issn)
    if issn != 'nan':
        return issn[0:4] + '-' + issn[4:]
    else:
        return np.nan

In [13]:
df_all['p_issn'] = df_all['ISSN'].apply(process_issn)
df_all['e_issn'] = df_all['EISSN'].apply(process_issn)
df_all['service'] = 'scopus'
df_all['internal_identifier'] = df_all['Sourcerecord ID']

In [14]:
df_all['edition'] = '261028'

In [15]:
df_all_slice = df_all[['service', 'edition', 'internal_identifier', 
                       'journal_title', 'p_issn', 'e_issn', 
                       'indexing_status', 'notes_on_indexing_status']].copy()

In [16]:
file_prefix = '261028_scopus'
df_all_slice.to_csv('../data/' + file_prefix + '.csv', index=False)
df_all_slice.to_parquet('../data/' + file_prefix + '.parquet')